# [rsyncd-ssh-server](https://github.com/login/device)

Simple notes for setting up and testing this project.

References:
- https://linux.die.net/man/5/rsyncd.conf
- https://superuser.com/questions/243656/how-to-configure-and-use-rsyncd


In [ ]:
# Password Generator
printf 'username:%s\n' "$(openssl rand -base64 64 | head -c 64 | tr -d '\n' | tr -dc 'A-Za-z0-9')"
openssl rand -base64 68 | tr -dc 'A-Za-z0-9' | head -c 64

In [ ]:
# Password Generator (better)
COUNT=12
LENGTH=64
FILTER='A-Za-z0-9' # alnum:'A-Za-z0-9'  # low:'a-z0-9'  # hex:'A-Fa-f0-9'

printf 'Password Generator | COUNT=%s | LENGTH=%s | FILTER=%s\n' \
  "$COUNT" "$LENGTH" "$FILTER"

for ((i=1; i<=COUNT; i++)); do
  openssl rand -base64 "$((LENGTH + 12))" | tr -dc 'A-Za-z0-9' | head -c "$LENGTH"
  echo
done

## Docker & Docker Compose

Edit the example files for your environment. Rename them if you want the conventional names, for example `docker-compose.example.env` to `.env` and `secrets/rsyncd.example.secrets` to `secrets/rsyncd.secrets`.


### Docker Context

In [ ]:
# Use Context
# docker context use default
docker context use nas-03

## Docker Build and Deploy Container

### Example

In [ ]:
# Build (example)
docker compose --progress plain --profile '*' --file ../docker-compose.yaml --env-file ../docker-compose.example.env build

In [ ]:
# Build & Start (rsyncd only) (example)
docker compose --file ../docker-compose.yaml --env-file ../docker-compose.example.env up --detach --build

In [ ]:
# Build & Start (rsyncd and ssh) (example)
docker compose --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.example.env up --detach --build

### NAS-03 (UNRAID)

In [ ]:
# Build
docker compose --progress plain --profile '*' --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env build

In [ ]:
# Build & Start (rsyncd only)
docker compose --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

In [ ]:
# Build & Start (rsyncd and ssh)
docker compose --profile ssh --file ../docker-compose.yaml --env-file ../docker-compose.nas-03_unraid.env up --detach --build

## Synology Hyper Backup

Plain rsyncd:
- Server type: `rsync-compatible server`
- Port: `873` or `RSYNCD_PORT`
- Username: a user allowed by the target module
- Backup module: the module name

Encrypted over SSH:
- Server type: `rsync-compatible server`
- Transfer encryption: `On`
- Port: `2222` or `RSYNC_SSH_PORT`
- Username: the SSH user for that module
- Backup module: the rsync module name, for example `nas-01`
- Directory: leave blank unless you intentionally want a subfolder inside that module

The Synology SSH flow sends `rsync --server --daemon .`, so the SSH container must serve rsync modules for the logged-in SSH user.


## rsyncd Tests

### Examples

In [ ]:
# List modules from inside the rsyncd container
docker exec rsyncd-server rsync rsync://127.0.0.1/

In [ ]:
# List modules from another machine
rsync rsync://SERVER_IP_HOSTNAME:873/

In [ ]:
# List files in an example module using a password file
chmod 600 ../secrets/rsync.example.pass
cat ../secrets/rsync.example.pass
rsync --password-file=../secrets/rsync.example.pass rsync://servers_user@SERVER_IP_HOSTNAME:873/servers-main/

### NAS-03 (UNRAID)

In [ ]:
# List modules from inside the rsyncd container
docker exec rsyncd-server rsync rsync://127.0.0.1/

In [ ]:
# List modules from another machine
rsync rsync://nas-03.internal:873/

In [ ]:
# List files in an example module using a password file
PASS_FILE="../secrets/rsync.test.pass"
chmod 600 "${PASS_FILE}"
# cat "${PASS_FILE}"
rsync --password-file="${PASS_FILE}" rsync://test_user@nas-03.internal:873/testing/

## Synology SSH Tests


In [ ]:
# List modules for one SSH user
SSH_PORT=2222
SSH_USER=nas-01
SSH_HOST=nas-03.internal
rsync -e "ssh -p ${SSH_PORT}" ${SSH_USER}@${SSH_HOST}::

In [ ]:
# List one module over SSH
SSH_PORT=2222
SSH_USER=nas-01
SSH_HOST=nas-03.internal
MODULE_NAME=nas-01
rsync -e "ssh -p ${SSH_PORT}" ${SSH_USER}@${SSH_HOST}::${MODULE_NAME}

## Troubleshooting

- If SSH authentication fails, manually type the password and confirm it matches the user in `RSYNCD_USERS` or the active secrets file.
- If SSH login works but Synology cannot find modules, test the same flow manually with `rsync -e "ssh -p 2222" USER@HOST::`.
- If you see `rsync: did not see server greeting`, the SSH service is not successfully handling `rsync --server --daemon .` yet.
- A simple working pattern is one SSH user per backup module, for example `nas-01` -> `nas-01`.
- After debugging, keep `RSYNC_SSH_TRACE_COMMANDS=false` and `RSYNC_SSH_LOG_LEVEL=INFO`.


In [ ]:
# Useful logs
docker logs --tail 200 rsyncd-server-ssh
docker logs --tail 200 rsyncd-server